## 1. Khởi tạo môi trường

Thiết lập project root và Python path để notebook có thể sử dụng các module nội bộ của project.


In [1]:
from pathlib import Path
import sys

PROJ_ROOT = Path.cwd().parent

if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from features.engineering import add_features
from ml_challenge.config import INTERIM_DATA_DIR

## 2. Đọc cleaned dataset


In [3]:
train = pd.read_csv(INTERIM_DATA_DIR / "train_clean.csv")
test = pd.read_csv(INTERIM_DATA_DIR / "test_clean.csv")

print("Train:", train.shape)
print("Test :", test.shape)

Train: (699960, 26)
Test : (299984, 25)


In [4]:
TARGET = "is_fraud"
ID_COL = "id"

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]

X_test = test.drop(columns=[ID_COL])

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (699960, 24)
y: (699960,)
X_test: (299984, 24)


In [5]:
X_fe = add_features(X)
X_test_fe = add_features(X_test)

print("Original features:", X.shape[1])
print("Features after engineering:", X_fe.shape[1])

Original features: 24
Features after engineering: 29


## 3. Feature Engineering

In [6]:
print("\nFeature list:")
for i, col in enumerate(X_fe.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nTotal candidate features:", X_fe.shape[1])


Feature list:
01. amount_usd
02. hours_since_last_txn
03. txn_count_last_24h
04. distance_from_home_km
05. card_age_months
06. customer_age
07. account_balance_usd
08. cvv_retry_count
09. velocity_score
10. time_of_day_hour
11. day_of_week
12. merchant_risk_score
13. prior_disputes
14. merchant_category
15. card_type
16. auth_method
17. channel
18. device_type
19. is_foreign_transaction
20. is_new_merchant
21. used_vpn
22. ip_country_mismatch
23. billing_shipping_mismatch
24. is_ai_generated_scam_attempt
25. red_flag_count
26. amount_balance_ratio
27. txn_rate_24h
28. risk_velocity_interaction
29. risk_flag_interaction

Total candidate features: 29


## 4. Kiểm tra kiểu dữ liệu

In [7]:
print("Feature dtypes:")
print(X_fe.dtypes)

print("\nNumerical features:")
print(X_fe.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical features:")
print(X_fe.select_dtypes(include=["object", "category"]).columns.tolist())

Feature dtypes:
amount_usd                      float64
hours_since_last_txn            float64
txn_count_last_24h                int64
distance_from_home_km           float64
card_age_months                   int64
customer_age                      int64
account_balance_usd             float64
cvv_retry_count                   int64
velocity_score                  float64
time_of_day_hour                  int64
day_of_week                       int64
merchant_risk_score             float64
prior_disputes                    int64
merchant_category                   str
card_type                           str
auth_method                         str
channel                             str
device_type                         str
is_foreign_transaction             bool
is_new_merchant                    bool
used_vpn                           bool
ip_country_mismatch                bool
billing_shipping_mismatch          bool
is_ai_generated_scam_attempt       bool
red_flag_count          

C:\Users\Uht\AppData\Local\Temp\ipykernel_14128\1888416786.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(X_fe.select_dtypes(include=["object", "category"]).columns.tolist())


## 5. Kiểm tra phân bố target

In [8]:
print("Target distribution:")
print(y.value_counts())

print("\nTarget proportion:")
print(y.value_counts(normalize=True))

print("\nFraud rate:")
print(y.mean())

Target distribution:
is_fraud
0.0    688271
1.0     11689
Name: count, dtype: int64

Target proportion:
is_fraud
0.0    0.9833
1.0    0.0167
Name: proportion, dtype: float64

Fraud rate:
0.016699525687182123


## 6. Tạo Feature Selection split

Feature Selection được thực hiện dựa trên FS train; validation được giữ lại để đánh giá candidate feature sets.


In [9]:
from sklearn.model_selection import train_test_split

X_fs_train, X_fs_valid, y_fs_train, y_fs_valid = train_test_split(
    X_fe,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("FS train:", X_fs_train.shape)
print("FS valid:", X_fs_valid.shape)

print("\nFS train target:")
print(y_fs_train.value_counts(normalize=True))

print("\nFS valid target:")
print(y_fs_valid.value_counts(normalize=True))

FS train: (559968, 29)
FS valid: (139992, 29)

FS train target:
is_fraud
0.0    0.983301
1.0    0.016699
Name: proportion, dtype: float64

FS valid target:
is_fraud
0.0    0.983299
1.0    0.016701
Name: proportion, dtype: float64


## 7. Mutual Information — đánh giá mức độ liên quan với target

**Mục tiêu:** đo mức độ phụ thuộc giữa từng feature và biến mục tiêu `is_fraud`.

Mutual Information (MI) được thực hiện riêng cho:
- Numerical features
- Categorical features
- Boolean features

**Nguyên tắc:** MI cao cho thấy feature có nhiều thông tin liên quan đến target hơn. Tuy nhiên, MI chỉ đánh giá **relevance**, chưa cho biết feature có thực sự cải thiện mô hình hay có bị dư thừa với feature khác hay không.

> **Lưu ý:** Không loại feature chỉ dựa trên MI. Kết quả MI sẽ được kết hợp với các phương pháp khác ở các bước sau.


#### 7.1. Mutual Information cho Numerical features

In [10]:
from sklearn.feature_selection import mutual_info_classif

numeric_cols = X_fs_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Number of numerical features:", len(numeric_cols))
print(numeric_cols)

Number of numerical features: 18
['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'red_flag_count', 'amount_balance_ratio', 'txn_rate_24h', 'risk_velocity_interaction', 'risk_flag_interaction']


In [11]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X_fs_train[numeric_cols],
    y_fs_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "feature": numeric_cols,
    "mi_score": mi_scores
}).sort_values("mi_score", ascending=False)

print(mi_df.to_string(index=False))

                  feature  mi_score
       txn_count_last_24h  0.027496
           red_flag_count  0.025657
    risk_flag_interaction  0.016658
              day_of_week  0.015546
          cvv_retry_count  0.012836
risk_velocity_interaction  0.011122
      merchant_risk_score  0.008594
          card_age_months  0.007452
           prior_disputes  0.006936
           velocity_score  0.005680
         time_of_day_hour  0.005350
             customer_age  0.004665
             txn_rate_24h  0.002146
               amount_usd  0.001969
     hours_since_last_txn  0.001812
      account_balance_usd  0.001065
    distance_from_home_km  0.000310
     amount_balance_ratio  0.000277


##### Kiểm tra redundancy giữa numerical features

Sau khi đánh giá relevance, bước tiếp theo là kiểm tra xem các numerical features có chứa thông tin trùng lặp hay không.

Ở đây sử dụng **Spearman correlation** để phát hiện các feature có quan hệ đơn điệu mạnh.

Ngưỡng kiểm tra:

`|Spearman correlation| >= 0.80`

Sau khi đánh giá relevance, bước tiếp theo là kiểm tra xem các numerical features có chứa thông tin trùng lặp hay không.

In [12]:
corr_matrix = X_fs_train[numeric_cols].corr(method="spearman")

print(corr_matrix.round(3).to_string())

                           amount_usd  hours_since_last_txn  txn_count_last_24h  distance_from_home_km  card_age_months  customer_age  account_balance_usd  cvv_retry_count  velocity_score  time_of_day_hour  day_of_week  merchant_risk_score  prior_disputes  red_flag_count  amount_balance_ratio  txn_rate_24h  risk_velocity_interaction  risk_flag_interaction
amount_usd                      1.000                -0.003               0.009                  0.012            0.007        -0.000                0.007            0.007           0.010            -0.002        0.013                0.148           0.001          -0.004                 0.739         0.007                      0.077                  0.005
hours_since_last_txn           -0.003                 1.000              -0.042                 -0.006           -0.004        -0.011                0.006           -0.007          -0.444            -0.003        0.002               -0.008           0.001          -0.006             

#### 7.2. Mutual Information cho Categorical features

In [13]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif

categorical_cols = X_fs_train.select_dtypes(
    include=["str", "object", "category"]
).columns.tolist()

print("Categorical features:", categorical_cols)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_cat_train = encoder.fit_transform(X_fs_train[categorical_cols])

mi_cat_scores = mutual_info_classif(
    X_cat_train,
    y_fs_train,
    discrete_features=True,
    random_state=42
)

mi_cat_df = pd.DataFrame({
    "feature": categorical_cols,
    "mi_score": mi_cat_scores
}).sort_values("mi_score", ascending=False)

print("\nCategorical MI:")
print(mi_cat_df.to_string(index=False))

Categorical features: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']

Categorical MI:
          feature  mi_score
      auth_method  0.005330
merchant_category  0.003062
        card_type  0.000023
          channel  0.000012
      device_type  0.000009


##### All features ranked by Mutual Information

In [14]:
mi_all_df = pd.concat(
    [
        mi_df.assign(feature_type="numerical"),
        mi_cat_df.assign(feature_type="categorical")
    ],
    ignore_index=True
).sort_values("mi_score", ascending=False)

print("All features ranked by Mutual Information:")
print(mi_all_df.to_string(index=False))

All features ranked by Mutual Information:
                  feature  mi_score feature_type
       txn_count_last_24h  0.027496    numerical
           red_flag_count  0.025657    numerical
    risk_flag_interaction  0.016658    numerical
              day_of_week  0.015546    numerical
          cvv_retry_count  0.012836    numerical
risk_velocity_interaction  0.011122    numerical
      merchant_risk_score  0.008594    numerical
          card_age_months  0.007452    numerical
           prior_disputes  0.006936    numerical
           velocity_score  0.005680    numerical
         time_of_day_hour  0.005350    numerical
              auth_method  0.005330  categorical
             customer_age  0.004665    numerical
        merchant_category  0.003062  categorical
             txn_rate_24h  0.002146    numerical
               amount_usd  0.001969    numerical
     hours_since_last_txn  0.001812    numerical
      account_balance_usd  0.001065    numerical
    distance_from_home_km 

#### 7.3. Mutual Information cho Boolean features

In [15]:
boolean_cols = X_fs_train.select_dtypes(include=["bool"]).columns.tolist()

print("Boolean features:", boolean_cols)
print("Number of boolean features:", len(boolean_cols))

Boolean features: ['is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']
Number of boolean features: 6


In [16]:
boolean_cols = X_fs_train.select_dtypes(
    include=["bool"]
).columns.tolist()

mi_bool_scores = mutual_info_classif(
    X_fs_train[boolean_cols],
    y_fs_train,
    discrete_features=True,
    random_state=42
)

mi_bool_df = pd.DataFrame({
    "feature": boolean_cols,
    "mi_score": mi_bool_scores
}).sort_values("mi_score", ascending=False)

print("Boolean MI:")
print(mi_bool_df.to_string(index=False))

Boolean MI:
                     feature  mi_score
         ip_country_mismatch  0.004129
             is_new_merchant  0.003621
      is_foreign_transaction  0.002726
   billing_shipping_mismatch  0.002127
                    used_vpn  0.001458
is_ai_generated_scam_attempt  0.000701


#### 7.4. Tổng hợp Mutual Information

In [17]:
mi_29_df = pd.concat(
    [
        mi_df.assign(feature_type="numerical"),
        mi_cat_df.assign(feature_type="categorical"),
        mi_bool_df.assign(feature_type="boolean")
    ],
    ignore_index=True
).sort_values("mi_score", ascending=False)

print("Total features:", len(mi_29_df))
print("\nAll 29 features ranked by Mutual Information:")
print(mi_29_df.to_string(index=False))

Total features: 29

All 29 features ranked by Mutual Information:
                     feature  mi_score feature_type
          txn_count_last_24h  0.027496    numerical
              red_flag_count  0.025657    numerical
       risk_flag_interaction  0.016658    numerical
                 day_of_week  0.015546    numerical
             cvv_retry_count  0.012836    numerical
   risk_velocity_interaction  0.011122    numerical
         merchant_risk_score  0.008594    numerical
             card_age_months  0.007452    numerical
              prior_disputes  0.006936    numerical
              velocity_score  0.005680    numerical
            time_of_day_hour  0.005350    numerical
                 auth_method  0.005330  categorical
                customer_age  0.004665    numerical
         ip_country_mismatch  0.004129      boolean
             is_new_merchant  0.003621      boolean
           merchant_category  0.003062  categorical
      is_foreign_transaction  0.002726      boolea

## 8. Model-based Feature Importance

MI chỉ xem xét quan hệ giữa từng feature và target. Tiếp theo, sử dụng Random Forest để đánh giá feature importance trong bối cảnh mô hình dự đoán.

### Quy trình

1. Tách numerical, categorical và boolean features.
2. One-hot encode categorical features.
3. Giữ nguyên numerical và boolean features.
4. Huấn luyện Random Forest trên FS-train.
5. Lấy feature importance.
6. Aggregate các encoded features về feature gốc.

Random Forest được cấu hình với:
- `n_estimators = 200`
- `class_weight = "balanced"`
- `random_state = 42`

`class_weight="balanced"` được sử dụng vì target bị mất cân bằng mạnh.

#### 8.1. Encoding cho model-based analysis

Random Forest cần dữ liệu số. Numerical và boolean được giữ nguyên; categorical được One-Hot Encoding.

Kết quả: 29 feature gốc được biểu diễn thành 59 encoded features.

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_cols = X_fs_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_fs_train.select_dtypes(
    include=["str", "object", "category"]
).columns.tolist()

boolean_cols = X_fs_train.select_dtypes(
    include=["bool"]
).columns.tolist()

print("Numerical:", len(numeric_cols))
print("Categorical:", len(categorical_cols))
print("Boolean:", len(boolean_cols))

preprocessor_fs = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ), categorical_cols),
        ("bool", "passthrough", boolean_cols)
    ]
)

X_fs_train_encoded = preprocessor_fs.fit_transform(X_fs_train)
X_fs_valid_encoded = preprocessor_fs.transform(X_fs_valid)

print("\nEncoded train shape:", X_fs_train_encoded.shape)
print("Encoded valid shape:", X_fs_valid_encoded.shape)

Numerical: 18
Categorical: 5
Boolean: 6

Encoded train shape: (559968, 59)
Encoded valid shape: (139992, 59)


#### 8.2 Huấn luyện Random Forest để đánh giá importance

Random Forest được dùng ở đây **như một công cụ đánh giá feature importance**, chưa phải model cuối cùng của project.

Sau one-hot encoding, 29 feature gốc được biểu diễn thành 59 encoded features.

Feature importance ban đầu được tính ở mức encoded feature, sau đó sẽ được cộng gộp về 29 feature gốc.

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_fs = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",    # dùng balanced để xử lý dữ liệu bị lệch
    max_depth=None
)

rf_fs.fit(X_fs_train_encoded, y_fs_train)

print("Random Forest training completed.")

Random Forest training completed.


#### 8.3. Importance ở mức encoded feature

Random Forest trả importance cho từng encoded column. Các encoded columns thuộc cùng một categorical feature sẽ được cộng lại về feature gốc ở bước tiếp theo.

In [20]:
feature_names_encoded = preprocessor_fs.get_feature_names_out()

print("Number of encoded features:", len(feature_names_encoded))

importance_df = pd.DataFrame({
    "encoded_feature": feature_names_encoded,
    "importance": rf_fs.feature_importances_
})

print(importance_df.sort_values(
    "importance",
    ascending=False
).head(20).to_string(index=False))

Number of encoded features: 59
                   encoded_feature  importance
    num__risk_velocity_interaction    0.116877
              num__cvv_retry_count    0.112363
        num__risk_flag_interaction    0.106442
          num__merchant_risk_score    0.076867
               num__red_flag_count    0.061539
               num__velocity_score    0.054904
cat__auth_method_No Authentication    0.050918
                 num__customer_age    0.042710
                 num__txn_rate_24h    0.029621
                   num__amount_usd    0.026613
         num__hours_since_last_txn    0.026208
        num__distance_from_home_km    0.025823
          num__account_balance_usd    0.025342
         num__amount_balance_ratio    0.025330
              num__card_age_months    0.020692
             num__time_of_day_hour    0.019828
             bool__is_new_merchant    0.018419
         bool__ip_country_mismatch    0.016086
           num__txn_count_last_24h    0.014978
                  num__day_of

#### 8.4. Aggregate về feature gốc

Các encoded columns thuộc cùng một feature gốc được cộng importance lại để đưa kết quả về cùng đơn vị 29 original features.


In [21]:
def get_original_feature(encoded_name):
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "", 1)
    elif encoded_name.startswith("bool__"):
        return encoded_name.replace("bool__", "", 1)
    elif encoded_name.startswith("cat__"):
        cat_name = encoded_name.replace("cat__", "", 1)

        for col in categorical_cols:
            if cat_name.startswith(col + "_"):
                return col

        return cat_name

    return encoded_name


importance_df["original_feature"] = (
    importance_df["encoded_feature"]
    .apply(get_original_feature)
)

feature_importance_df = (
    importance_df
    .groupby("original_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

print("Number of original features:", len(feature_importance_df))
print("\nFeature importance by original feature:")
print(feature_importance_df.to_string(index=False))

Number of original features: 29

Feature importance by original feature:
            original_feature  importance
   risk_velocity_interaction    0.116877
             cvv_retry_count    0.112363
       risk_flag_interaction    0.106442
         merchant_risk_score    0.076867
                 auth_method    0.071537
              red_flag_count    0.061539
              velocity_score    0.054904
                customer_age    0.042710
                txn_rate_24h    0.029621
           merchant_category    0.026942
                  amount_usd    0.026613
        hours_since_last_txn    0.026208
       distance_from_home_km    0.025823
         account_balance_usd    0.025342
        amount_balance_ratio    0.025330
             card_age_months    0.020692
            time_of_day_hour    0.019828
                 device_type    0.018540
             is_new_merchant    0.018419
         ip_country_mismatch    0.016086
          txn_count_last_24h    0.014978
                     chan

## 9. Kết hợp MI và Random Forest importance

Hai phương pháp cung cấp hai góc nhìn:

| Phương pháp | Câu hỏi chính |
|---|---|
| Mutual Information | Feature có liên quan đến target đến mức nào? |
| Random Forest importance | Feature đóng góp thế nào trong Random Forest? |

Để tránh phụ thuộc vào một thước đo duy nhất, rank của mỗi feature theo hai phương pháp được kết hợp thành `rank_avg`.

**Mục đích:** tạo cơ sở cho candidate feature sets; chưa phải quyết định loại feature.

#### Feature Selection Summary

Kết hợp MI và Random Forest importance vào một bảng duy nhất. Bảng này là cơ sở cho ranking tổng hợp và xây dựng candidate feature sets.

In [22]:
fs_summary = mi_29_df.merge(
    feature_importance_df,
    left_on="feature",
    right_on="original_feature",
    how="left"
).drop(columns=["original_feature"])

fs_summary = fs_summary.rename(
    columns={"importance": "rf_importance"}
)

fs_summary = fs_summary.sort_values(
    "rf_importance",
    ascending=False
)

print("Feature Selection Summary:")
print(fs_summary.to_string(index=False))

Feature Selection Summary:
                     feature  mi_score feature_type  rf_importance
   risk_velocity_interaction  0.011122    numerical       0.116877
             cvv_retry_count  0.012836    numerical       0.112363
       risk_flag_interaction  0.016658    numerical       0.106442
         merchant_risk_score  0.008594    numerical       0.076867
                 auth_method  0.005330  categorical       0.071537
              red_flag_count  0.025657    numerical       0.061539
              velocity_score  0.005680    numerical       0.054904
                customer_age  0.004665    numerical       0.042710
                txn_rate_24h  0.002146    numerical       0.029621
           merchant_category  0.003062  categorical       0.026942
                  amount_usd  0.001969    numerical       0.026613
        hours_since_last_txn  0.001812    numerical       0.026208
       distance_from_home_km  0.000310    numerical       0.025823
         account_balance_usd  0.001

## 10. Xây dựng ranking tổng hợp

Tính:
- `mi_rank`: thứ hạng theo Mutual Information.
- `rf_rank`: thứ hạng theo Random Forest importance.
- `rank_avg`: trung bình của hai thứ hạng.

Feature có `rank_avg` thấp hơn được ưu tiên hơn theo hai tiêu chí kết hợp.

> Ranking này chỉ hỗ trợ xây dựng candidate sets. Quyết định cuối cùng vẫn dựa trên validation performance.

In [23]:
fs_summary["mi_rank"] = (
    fs_summary["mi_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

fs_summary["rf_rank"] = (
    fs_summary["rf_importance"]
    .rank(method="min", ascending=False)
    .astype(int)
)

fs_summary["rank_avg"] = (
    fs_summary["mi_rank"] + fs_summary["rf_rank"]
) / 2

fs_summary = fs_summary.sort_values("rank_avg")

print("Feature Selection Ranking:")
print(
    fs_summary[
        [
            "feature",
            "mi_score",
            "rf_importance",
            "mi_rank",
            "rf_rank",
            "rank_avg"
        ]
    ].to_string(index=False)
)

Feature Selection Ranking:
                     feature  mi_score  rf_importance  mi_rank  rf_rank  rank_avg
       risk_flag_interaction  0.016658       0.106442        3        3       3.0
   risk_velocity_interaction  0.011122       0.116877        6        1       3.5
             cvv_retry_count  0.012836       0.112363        5        2       3.5
              red_flag_count  0.025657       0.061539        2        6       4.0
         merchant_risk_score  0.008594       0.076867        7        4       5.5
                 auth_method  0.005330       0.071537       12        5       8.5
              velocity_score  0.005680       0.054904       10        7       8.5
                customer_age  0.004665       0.042710       13        8      10.5
          txn_count_last_24h  0.027496       0.014978        1       21      11.0
             card_age_months  0.007452       0.020692        8       16      12.0
           merchant_category  0.003062       0.026942       16       10

## 11. Phân tích redundancy

Từ ma trận Spearman, chỉ giữ lại các cặp có:

`|correlation| >= 0.80`

Mục tiêu là xác định những feature có khả năng mang thông tin gần trùng nhau.

Mỗi cặp được đối chiếu với:
- MI score
- Random Forest importance
- MI rank
- RF rank
- Average rank

Như vậy, quyết định loại feature không chỉ dựa trên correlation.

#### 11.1. Tìm các cặp tương quan cao

Chỉ các cặp có `abs(Spearman correlation) >= 0.80` được đưa vào danh sách redundancy candidates.

In [24]:
corr_threshold = 0.80

corr_pairs = []

for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        corr_value = corr_matrix.iloc[i, j]

        if abs(corr_value) >= corr_threshold:
            corr_pairs.append({
                "feature_1": numeric_cols[i],
                "feature_2": numeric_cols[j],
                "spearman_corr": corr_value,
                "abs_corr": abs(corr_value)
            })

corr_pairs_df = pd.DataFrame(corr_pairs).sort_values(
    "abs_corr",
    ascending=False
)

print("Highly correlated feature pairs:")
print(corr_pairs_df.to_string(index=False))

Highly correlated feature pairs:
           feature_1                 feature_2  spearman_corr  abs_corr
      red_flag_count     risk_flag_interaction       0.990480  0.990480
hours_since_last_txn              txn_rate_24h      -0.856244  0.856244
      velocity_score risk_velocity_interaction       0.843275  0.843275


#### 11.2. Đối chiếu redundancy với importance

Không tự động loại feature chỉ vì correlation cao. Mỗi cặp được đối chiếu bằng MI, RF importance và rank tổng hợp trước khi xây dựng Candidate B.

In [25]:
redundancy_analysis = corr_pairs_df.merge(
    fs_summary[
        [
            "feature",
            "mi_score",
            "rf_importance",
            "mi_rank",
            "rf_rank",
            "rank_avg"
        ]
    ],
    left_on="feature_1",
    right_on="feature",
    how="left"
).rename(
    columns={
        "mi_score": "mi_1",
        "rf_importance": "rf_1",
        "mi_rank": "mi_rank_1",
        "rf_rank": "rf_rank_1",
        "rank_avg": "rank_avg_1"
    }
).drop(columns=["feature"])

redundancy_analysis = redundancy_analysis.merge(
    fs_summary[
        [
            "feature",
            "mi_score",
            "rf_importance",
            "mi_rank",
            "rf_rank",
            "rank_avg"
        ]
    ],
    left_on="feature_2",
    right_on="feature",
    how="left"
).rename(
    columns={
        "mi_score": "mi_2",
        "rf_importance": "rf_2",
        "mi_rank": "mi_rank_2",
        "rf_rank": "rf_rank_2",
        "rank_avg": "rank_avg_2"
    }
).drop(columns=["feature"])

print("Redundancy analysis:")
print(redundancy_analysis.to_string(index=False))

Redundancy analysis:
           feature_1                 feature_2  spearman_corr  abs_corr     mi_1     rf_1  mi_rank_1  rf_rank_1  rank_avg_1     mi_2     rf_2  mi_rank_2  rf_rank_2  rank_avg_2
      red_flag_count     risk_flag_interaction       0.990480  0.990480 0.025657 0.061539          2          6         4.0 0.016658 0.106442          3          3         3.0
hours_since_last_txn              txn_rate_24h      -0.856244  0.856244 0.001812 0.026208         21         12        16.5 0.002146 0.029621         18          9        13.5
      velocity_score risk_velocity_interaction       0.843275  0.843275 0.005680 0.054904         10          7         8.5 0.011122 0.116877          6          1         3.5


## 12. Xây dựng các candidate feature sets

Để kiểm chứng Feature Selection bằng thực nghiệm, xây dựng ba tập feature:

### Candidate A — Full
Giữ toàn bộ **29 features**. Đây là mốc tham chiếu.

### Candidate B — Reduced
Loại các feature được xem là dư thừa dựa trên các cặp correlation cao:
- `red_flag_count`
- `hours_since_last_txn`
- `velocity_score`

Còn **26 features**.

### Candidate C — Top 20
Giữ 20 features có `rank_avg` tốt nhất.

Còn **20 features**.

> Các candidate này là giả thuyết cần kiểm chứng bằng ROC-AUC, không phải kết luận cuối cùng.

In [26]:
# Candidate A: tất cả feature
features_all = X_fe.columns.tolist()

# Candidate B: loại các feature dư thừa
redundant_features = [
    "red_flag_count",
    "hours_since_last_txn",
    "velocity_score"
]

features_reduced = [
    col for col in features_all
    if col not in redundant_features
]

# Candidate C: Top 20 theo rank_avg
features_top20 = (
    fs_summary
    .sort_values("rank_avg")
    .head(20)["feature"]
    .tolist()
)

print("Candidate A - Full:")
print(f"Number of features: {len(features_all)}")
print(features_all)

print("\nCandidate B - Reduced:")
print(f"Number of features: {len(features_reduced)}")
print(features_reduced)

print("\nCandidate C - Top 20:")
print(f"Number of features: {len(features_top20)}")
print(features_top20)

Candidate A - Full:
Number of features: 29
['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt', 'red_flag_count', 'amount_balance_ratio', 'txn_rate_24h', 'risk_velocity_interaction', 'risk_flag_interaction']

Candidate B - Reduced:
Number of features: 26
['amount_usd', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_m

## 13. Benchmark 3 candidates feature sets

#### 13.1. Hàm benchmark

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


def prepare_features(X_train, X_valid, feature_list):
    X_train_subset = X_train[feature_list]
    X_valid_subset = X_valid[feature_list]

    numeric_cols_subset = X_train_subset.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_cols_subset = X_train_subset.select_dtypes(
        include=["str", "object", "category"]
    ).columns.tolist()

    boolean_cols_subset = X_train_subset.select_dtypes(
        include=["bool"]
    ).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                "passthrough",
                numeric_cols_subset
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                categorical_cols_subset
            ),
            (
                "bool",
                "passthrough",
                boolean_cols_subset
            )
        ]
    )

    X_train_encoded = preprocessor.fit_transform(X_train_subset)
    X_valid_encoded = preprocessor.transform(X_valid_subset)

    return (
        X_train_encoded,
        X_valid_encoded,
        preprocessor
    )


def evaluate_candidate(
    candidate_name,
    feature_list,
    X_train,
    X_valid,
    y_train,
    y_valid
):
    print("=" * 60)
    print(candidate_name)
    print("Number of original features:", len(feature_list))

    X_train_encoded, X_valid_encoded, preprocessor = prepare_features(
        X_train,
        X_valid,
        feature_list
    )

    print("Encoded train shape:", X_train_encoded.shape)
    print("Encoded valid shape:", X_valid_encoded.shape)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
        max_depth=None
    )

    model.fit(
        X_train_encoded,
        y_train
    )

    y_valid_proba = model.predict_proba(
        X_valid_encoded
    )[:, 1]

    auc = roc_auc_score(
        y_valid,
        y_valid_proba
    )

    print("Validation ROC-AUC:", auc)

    return {
        "candidate": candidate_name,
        "n_features": len(feature_list),
        "n_encoded_features": X_train_encoded.shape[1],
        "roc_auc": auc
    }

#### 13.2. Chạy candidates

In [28]:
result_a = evaluate_candidate(
    "Candidate A - Full",
    features_all,
    X_fs_train,
    X_fs_valid,
    y_fs_train,
    y_fs_valid
)

Candidate A - Full
Number of original features: 29
Encoded train shape: (559968, 59)
Encoded valid shape: (139992, 59)
Validation ROC-AUC: 0.9307874799168862


In [29]:
result_b = evaluate_candidate(
    "Candidate B - Reduced",
    features_reduced,
    X_fs_train,
    X_fs_valid,
    y_fs_train,
    y_fs_valid
)

Candidate B - Reduced
Number of original features: 26
Encoded train shape: (559968, 56)
Encoded valid shape: (139992, 56)
Validation ROC-AUC: 0.930644218641542


In [30]:
result_c = evaluate_candidate(
    "Candidate C - Top 20",
    features_top20,
    X_fs_train,
    X_fs_valid,
    y_fs_train,
    y_fs_valid
)

Candidate C - Top 20
Number of original features: 20
Encoded train shape: (559968, 35)
Encoded valid shape: (139992, 35)
Validation ROC-AUC: 0.9267801351855236


#### 13.3. So sánh kết quả Feature Selection

So sánh số feature, số encoded feature, ROC-AUC và thay đổi ROC-AUC so với Full. **Quyết định cuối cùng dựa trên validation ROC-AUC**, không chỉ dựa trên importance.

In [31]:
results_fs = pd.DataFrame([
    result_a,
    result_b,
    result_c
])

results_fs["auc_change_vs_full"] = (
    results_fs["roc_auc"] - result_a["roc_auc"]
)

results_fs["feature_reduction"] = (
    29 - results_fs["n_features"]
)

print("Feature Selection Experiment Results:")
print(results_fs.to_string(index=False))

Feature Selection Experiment Results:
            candidate  n_features  n_encoded_features  roc_auc  auc_change_vs_full  feature_reduction
   Candidate A - Full          29                  59 0.930787            0.000000                  0
Candidate B - Reduced          26                  56 0.930644           -0.000143                  3
 Candidate C - Top 20          20                  35 0.926780           -0.004007                  9


## 14. Kiểm chứng bằng Permutation Importance

Sau khi benchmark các candidate, sử dụng **Permutation Importance** để kiểm tra thêm contribution của feature đối với ROC-AUC.

### Ý tưởng

Với một feature:
1. Giữ nguyên model.
2. Xáo trộn giá trị của feature trên validation set.
3. Đo lại ROC-AUC.
4. Quan sát mức thay đổi.

Nếu việc xáo trộn feature làm ROC-AUC giảm đáng kể, feature đó có đóng góp hữu ích cho model.

Permutation Importance được tính trên encoded features trước, sau đó aggregate về feature gốc.

> Đây là bước kiểm chứng bổ sung, không thay thế cho validation benchmark của các candidate sets.

#### 14.1. Train lại Candidate A để kiểm tra Permutation Importance

Candidate A được train lại với cùng split, 29 features, Random Forest configuration và `random_state=42`. ROC-AUC được kiểm tra lại trước khi chạy Permutation Importance.

In [32]:
from sklearn.inspection import permutation_importance

# Chuẩn bị Candidate A - Full
X_fs_train_a = X_fs_train[features_all]
X_fs_valid_a = X_fs_valid[features_all]

numeric_cols_a = X_fs_train_a.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols_a = X_fs_train_a.select_dtypes(
    include=["str", "object", "category"]
).columns.tolist()

boolean_cols_a = X_fs_train_a.select_dtypes(
    include=["bool"]
).columns.tolist()

preprocessor_a = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_cols_a
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_cols_a
        ),
        (
            "bool",
            "passthrough",
            boolean_cols_a
        )
    ]
)

X_train_a_encoded = preprocessor_a.fit_transform(X_fs_train_a)
X_valid_a_encoded = preprocessor_a.transform(X_fs_valid_a)

rf_a = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
    max_depth=None
)

rf_a.fit(
    X_train_a_encoded,
    y_fs_train
)

y_valid_a_proba = rf_a.predict_proba(
    X_valid_a_encoded
)[:, 1]

auc_a = roc_auc_score(
    y_fs_valid,
    y_valid_a_proba
)

print("Candidate A ROC-AUC:", auc_a)
print("Encoded train shape:", X_train_a_encoded.shape)
print("Encoded valid shape:", X_valid_a_encoded.shape)

Candidate A ROC-AUC: 0.9307874799168862
Encoded train shape: (559968, 59)
Encoded valid shape: (139992, 59)


#### 14.2. Permutation Importance ở mức encoded feature

Permutation Importance được tính trên validation set với scoring `ROC-AUC`. `n_repeats=3` được dùng để lấy mean importance và độ lệch chuẩn.

In [33]:
perm_result = permutation_importance(
    rf_a,
    X_valid_a_encoded,
    y_fs_valid,
    scoring="roc_auc",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

encoded_feature_names_a = (
    preprocessor_a.get_feature_names_out()
)

perm_encoded_df = pd.DataFrame({
    "encoded_feature": encoded_feature_names_a,
    "permutation_importance": perm_result.importances_mean,
    "std": perm_result.importances_std
})

print("Top 20 encoded features by Permutation Importance:")
print(
    perm_encoded_df
    .sort_values(
        "permutation_importance",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

Top 20 encoded features by Permutation Importance:
                   encoded_feature  permutation_importance      std
              num__cvv_retry_count                0.038718 0.000579
    num__risk_velocity_interaction                0.023729 0.001850
        num__risk_flag_interaction                0.014367 0.000703
cat__auth_method_No Authentication                0.012989 0.000729
                 num__customer_age                0.009290 0.001659
          num__merchant_risk_score                0.008018 0.000776
               num__velocity_score                0.007249 0.000829
               num__red_flag_count                0.006033 0.000360
                 num__txn_rate_24h                0.002587 0.000018
        cat__auth_method_Biometric                0.001980 0.000232
   bool__billing_shipping_mismatch                0.001066 0.000140
           num__txn_count_last_24h                0.001002 0.000530
        num__distance_from_home_km                0.000896 0.0002

#### 14.3. Aggregate Permutation Importance về feature gốc

Các encoded columns của cùng một categorical feature được cộng lại để thu được importance ở mức original feature.


In [34]:
def get_original_feature_perm(encoded_name):
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "", 1)

    elif encoded_name.startswith("bool__"):
        return encoded_name.replace("bool__", "", 1)

    elif encoded_name.startswith("cat__"):
        cat_name = encoded_name.replace("cat__", "", 1)

        for col in categorical_cols_a:
            if cat_name.startswith(col + "_"):
                return col

        return cat_name

    return encoded_name


perm_encoded_df["original_feature"] = (
    perm_encoded_df["encoded_feature"]
    .apply(get_original_feature_perm)
)

perm_feature_df = (
    perm_encoded_df
    .groupby("original_feature", as_index=False)
    ["permutation_importance"]
    .sum()
    .sort_values(
        "permutation_importance",
        ascending=False
    )
)

print("Permutation Importance by original feature:")
print(
    perm_feature_df.to_string(index=False)
)

Permutation Importance by original feature:
            original_feature  permutation_importance
             cvv_retry_count                0.038718
   risk_velocity_interaction                0.023729
                 auth_method                0.015820
       risk_flag_interaction                0.014367
                customer_age                0.009290
         merchant_risk_score                0.008018
              velocity_score                0.007249
              red_flag_count                0.006033
                txn_rate_24h                0.002587
   billing_shipping_mismatch                0.001066
          txn_count_last_24h                0.001002
           merchant_category                0.000997
       distance_from_home_km                0.000896
            time_of_day_hour                0.000815
         account_balance_usd                0.000787
        hours_since_last_txn                0.000774
                  amount_usd                0.000727
  

## 15. Tổng hợp kết quả Feature Selection

Bảng cuối cùng đặt cạnh nhau Mutual Information, Random Forest importance, các rank và Permutation Importance. Đây là bảng tổng hợp trước khi chốt feature set.


In [35]:
final_fs_comparison = (
    fs_summary[
        [
            "feature",
            "mi_score",
            "rf_importance",
            "mi_rank",
            "rf_rank",
            "rank_avg"
        ]
    ]
    .merge(
        perm_feature_df,
        left_on="feature",
        right_on="original_feature",
        how="left"
    )
    .drop(columns=["original_feature"])
    .sort_values(
        "permutation_importance",
        ascending=False
    )
)

print("Final Feature Selection Comparison:")
print(
    final_fs_comparison.to_string(index=False)
)

Final Feature Selection Comparison:
                     feature  mi_score  rf_importance  mi_rank  rf_rank  rank_avg  permutation_importance
             cvv_retry_count  0.012836       0.112363        5        2       3.5                0.038718
   risk_velocity_interaction  0.011122       0.116877        6        1       3.5                0.023729
                 auth_method  0.005330       0.071537       12        5       8.5                0.015820
       risk_flag_interaction  0.016658       0.106442        3        3       3.0                0.014367
                customer_age  0.004665       0.042710       13        8      10.5                0.009290
         merchant_risk_score  0.008594       0.076867        7        4       5.5                0.008018
              velocity_score  0.005680       0.054904       10        7       8.5                0.007249
              red_flag_count  0.025657       0.061539        2        6       4.0                0.006033
          

## 16. Kết luận Feature Selection

#### Kết quả thực nghiệm

| Candidate | Original features | ROC-AUC |
|---|---:|---:|
| Full | 29 | **0.930787** |
| Reduced | 26 | 0.930644 |
| Top 20 | 20 | 0.926780 |

Candidate Full đạt ROC-AUC cao nhất trong ba candidate.

#### Quyết định

**Final Feature Set: 29 features**

Các candidate giảm feature không cho thấy cải thiện validation ROC-AUC, vì vậy toàn bộ 29 candidate features được giữ lại cho bước Modeling.

Correlation cao không được xem là lý do đủ mạnh để tự động loại feature; một số feature có correlation cao vẫn có đóng góp dương và việc loại chúng làm ROC-AUC giảm nhẹ.

#### Kết quả của notebook

- Đánh giá relevance bằng Mutual Information.
- Đánh giá model-based importance bằng Random Forest.
- Kiểm tra redundancy bằng Spearman correlation.
- Kiểm chứng contribution bằng Permutation Importance.
- Benchmark trực tiếp các candidate feature sets bằng ROC-AUC.

**Kết luận cuối:** sử dụng toàn bộ 29 features cho bước Modeling tiếp theo.